# Home Assignment
## Two molecules a graph model cannot tell apart



---

### The chemistry

Compare the carbon skeletons of two hydrocarbons, hydrogens suppressed throughout.

```
        A: cyclohexane skeleton              B: two cyclopropane skeletons

               0                                   0            3
            /     \                               / \          / \
           1       5                             1---2        4---5
           |       |
           2       4
            \     /
               3
```

Both graphs have **six carbon atoms** and **six carbon-carbon bonds**, and every atom
carries the same initial label, `"C"`. Chemically they could hardly be more different:
one is a strain-free chair, the other carries roughly 115 kJ/mol of ring strain **per ring**.

Your task is to establish exactly what a standard message-passing model can and cannot
perceive here, and then to fix it.



In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

def show(name, A):
    """Print an adjacency matrix with its degree column."""
    print(f'{name}   (degrees on the right)')
    for i, row in enumerate(A.astype(int)):
        print('  ' + ' '.join(str(v) for v in row) + f'   | d_{i} = {int(row.sum())}')
    print(f'  bonds = {int(A.sum() // 2)},  atoms = {A.shape[0]}')
    print()


---
## Task 1. Write both adjacency matrices  &nbsp;&nbsp;

Fill in `A_A` and `A_B` using the atom labels in the diagram above.

Reminders:
- The adjacency matrix is $A_{ij}=1$ when atoms $i$ and $j$ are bonded, and $0$ otherwise.
- A molecular graph is undirected, so $A$ must be **symmetric**.
- There are no self-bonds, so the diagonal is zero.
- In **B** the two rings are *not* connected to each other. Atoms 0,1,2 form one ring
  and atoms 3,4,5 the other.


In [2]:
# Ring A: 0-1, 1-2, 2-3, 3-4, 4-5, 5-0

A_A = np.zeros((6, 6))
A_A[0, 1] = A_A[1, 0] = 1.0
A_A[1, 2] = A_A[2, 1] = 1.0
A_A[2, 3] = A_A[3, 2] = 1.0
A_A[3, 4] = A_A[4, 3] = 1.0
A_A[4, 5] = A_A[5, 4] = 1.0
A_A[5, 0] = A_A[0, 5] = 1.0

# Ring B: 0-1, 1-2, 2-0  and  3-4, 4-5, 5-3

A_B = np.zeros((6, 6))
A_B[0, 1] = A_B[1, 0] = 1.0
A_B[1, 2] = A_B[2, 1] = 1.0
A_B[2, 0] = A_B[0, 2] = 1.0
A_B[3, 4] = A_B[4, 3] = 1.0
A_B[4, 5] = A_B[5, 4] = 1.0
A_B[5, 3] = A_B[3, 5] = 1.0

show('A  (cyclohexane skeleton)', A_A)
show('B  (two cyclopropane skeletons)', A_B)


A  (cyclohexane skeleton)   (degrees on the right)
  0 1 0 0 0 1   | d_0 = 2
  1 0 1 0 0 0   | d_1 = 2
  0 1 0 1 0 0   | d_2 = 2
  0 0 1 0 1 0   | d_3 = 2
  0 0 0 1 0 1   | d_4 = 2
  1 0 0 0 1 0   | d_5 = 2
  bonds = 6,  atoms = 6

B  (two cyclopropane skeletons)   (degrees on the right)
  0 1 1 0 0 0   | d_0 = 2
  1 0 1 0 0 0   | d_1 = 2
  1 1 0 0 0 0   | d_2 = 2
  0 0 0 0 1 1   | d_3 = 2
  0 0 0 1 0 1   | d_4 = 2
  0 0 0 1 1 0   | d_5 = 2
  bonds = 6,  atoms = 6



In [3]:
# CHECK: structural properties only. These do not reveal the chemistry.
for name, A in [('A', A_A), ('B', A_B)]:
    assert A.shape == (6, 6),               f'{name}: must be 6x6'
    assert np.allclose(A, A.T),             f'{name}: must be symmetric'
    assert np.allclose(np.diag(A), 0),      f'{name}: diagonal must be zero'
    assert set(np.unique(A)) <= {0.0, 1.0}, f'{name}: entries must be 0 or 1'
    assert A.sum() // 2 == 6,               f'{name}: must have exactly 6 bonds'
assert not np.allclose(A_A, A_B), 'A and B must be different matrices'
print('Task 1 structural checks passed.')
print('Degree sequence A:', np.sort(A_A.sum(1)).astype(int))
print('Degree sequence B:', np.sort(A_B.sum(1)).astype(int))


Task 1 structural checks passed.
Degree sequence A: [2 2 2 2 2 2]
Degree sequence B: [2 2 2 2 2 2]


### YOUR ANSWER (Task 1)

State the degree of every atom in each graph, and say in one sentence what the degree
of a carbon atom means chemically.

*Write here:*

In **A** (cyclohexane skeleton) every atom has degree 2: d_0 = d_1 = d_2 = d_3 = d_4 = d_5 = 2.
In **B** (two cyclopropane rings) every atom also has degree 2: d_0 = ... = d_5 = 2.
Both graphs are therefore 2-regular with the identical degree sequence (2, 2, 2, 2, 2, 2).

Chemically, the degree of a carbon atom in the skeleton graph is the number of
carbon-carbon bonds it participates in, i.e. how many other ring carbons it is directly
bonded to (the remaining valences, up to 4, are filled by hydrogens that we suppress here).


---
## Task 2. Colour refinement  &nbsp;&nbsp;

**Do this by hand first, on paper.** The code is to check your hand work, not to replace it.

The 1-WL refinement rule is

$$c^{(k+1)}_i \;=\; \mathrm{HASH}\Big(c^{(k)}_i,\ \{\!\{\,c^{(k)}_j : j \in \mathcal{N}(i)\,\}\!\}\Big)$$

where $\{\!\{\cdot\}\!\}$ is a **multiset** (repetition matters, order does not) and
$\mathrm{HASH}$ assigns a fresh integer to each distinct signature it has seen.

Two graphs are declared **distinguishable** if at some round their *multisets of colours*
differ.

Complete the function below.


In [4]:
def wl_round(A, colours):
    """One round of 1-WL colour refinement.

    A       : (n, n) adjacency matrix
    colours : list of n integers, the current colours
    returns : list of n integers, the refined colours
    """
    n = len(colours)
    signatures = []
    for i in range(n):
        # Build the multiset of neighbour colours for atom i.
        neighbour_colours = tuple(sorted(colours[j] for j in range(n) if A[i, j] == 1))

        # The signature is the pair (own colour, neighbour multiset)
        signatures.append((colours[i], neighbour_colours))

    # relabel each distinct signature with a fresh integer (this part is done for you)
    table = {s: k for k, s in enumerate(sorted(set(signatures), key=str))}
    return [table[s] for s in signatures]


### Validate your implementation on a case with a known answer

Before trusting `wl_round` on A and B, test it on a pair where the answer is already known:
the carbon skeletons of **n-butane** (a chain) and **isobutane** (a central carbon with
three neighbours).

These two *are* distinguishable, and refinement should separate them after one round,
because their degree sequences differ.


In [5]:
# CHECK: a validation case with a known outcome. Do not edit.
A_nbutane  = np.array([[0,1,0,0],[1,0,1,0],[0,1,0,1],[0,0,1,0]], float)
A_isobutane = np.array([[0,1,1,1],[1,0,0,0],[1,0,0,0],[1,0,0,0]], float)

c1 = wl_round(A_nbutane,  [0, 0, 0, 0])
c2 = wl_round(A_isobutane, [0, 0, 0, 0])
print('n-butane  colours after 1 round:', sorted(c1))
print('isobutane colours after 1 round:', sorted(c2))
assert sorted(c1) != sorted(c2), (
    'Your wl_round does not separate n-butane from isobutane, but it should. '
    'Check that you are using a MULTISET of neighbour colours, not a set.')
print('\nValidation passed: wl_round behaves correctly on a known case.')


n-butane  colours after 1 round: [0, 0, 1, 1]
isobutane colours after 1 round: [0, 1, 1, 1]

Validation passed: wl_round behaves correctly on a known case.


In [6]:
# Now apply it to A and B. Two rounds, printed as a table.
cA = [0] * 6
cB = [0] * 6
print(f"{'round':<7}{'colours of A':<22}{'multiset A':<18}{'colours of B':<22}{'multiset B'}")
print('-' * 92)
for r in range(3):
    print(f'{r:<7}{str(cA):<22}{str(sorted(cA)):<18}{str(cB):<22}{str(sorted(cB))}')
    if r < 2:
        cA, cB = wl_round(A_A, cA), wl_round(A_B, cB)

print()
print('colour multisets identical at every round? ',
      all(sorted(a) == sorted(b) for a, b in [(cA, cB)]))


round  colours of A          multiset A        colours of B          multiset B
--------------------------------------------------------------------------------------------
0      [0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0][0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0]
1      [0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0][0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0]
2      [0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0][0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0]

colour multisets identical at every round?  True


### YOUR ANSWER (Task 2)

Reproduce your **hand** calculation here: the colour of each atom in each graph at rounds
0, 1 and 2, and the colour multiset of each graph at each round. Confirm that it agrees
with the code output above.

*Write here:*

**Round 0.** All atoms start with the same colour in both graphs: c_A = [0,0,0,0,0,0],
c_B = [0,0,0,0,0,0]. Multisets: {0,0,0,0,0,0} = {0,0,0,0,0,0}. Identical.

**Round 1.** Every atom in A has exactly two neighbours, and every one of those
neighbours currently has colour 0, so every atom's signature is (0, (0,0)) — the same
signature for all six atoms. Hence after HASHing, c_A = [0,0,0,0,0,0] again (relabelled
back to a single class). The same is true in B: every atom has exactly two neighbours,
both coloured 0, so every atom's signature is again (0, (0,0)), giving c_B = [0,0,0,0,0,0].
Multisets: {0,0,0,0,0,0} = {0,0,0,0,0,0}. Still identical.

**Round 2.** Nothing has changed: every atom in A still has two neighbours of colour 0
(the previous round produced one uniform colour again), so the signature (0,(0,0)) recurs
for every atom, and c_A stays [0,0,0,0,0,0]. Exactly the same argument applies to B. The
colour multisets remain identical: {0,0,0,0,0,0} = {0,0,0,0,0,0}.

This matches the printed table: because both graphs are 2-regular and start from a single
colour class, refinement can never split any atom off from the rest — every atom always
sees "one neighbour-multiset of size 2, all colour 0", in both graphs, forever. The
process has reached a fixed point at round 1 already, and A and B remain indistinguishable
at every round.


---
## Task 3. State the conclusion  &nbsp;&nbsp;

You now know what refinement does to A and B.


### YOUR ANSWER (Task 3)

Are A and B distinguishable by **any** message-passing model of the standard form

$$\bm{h}_i' = \phi_{\mathrm{upd}}\Big(\bm{h}_i,\ \bigoplus_{j \in \mathcal{N}(i)} \phi_{\mathrm{msg}}(\bm{h}_i, \bm{h}_j)\Big)?$$

Your justification must appeal to a **property of the two graphs**, not merely to the
outcome of your refinement. Two or three sentences.

Note carefully what the claim covers: it holds for every width, every depth, every choice
of $\phi$, and every amount of training data. Say why.

*Write here:*

No. Any standard message-passing GNN (any $\phi_{\mathrm{msg}}$, $\phi_{\mathrm{upd}}$,
aggregator $\bigoplus$, any width or depth, trained on any amount of data) is, on graphs
with identical, discrete initial node features, at most as powerful as 1-WL colour
refinement (the Xu et al. 2019 / Morris et al. 2019 result): if 1-WL cannot separate two
graphs, a GNN of this form produces the same multiset of node embeddings — and hence the
same graph-level readout — on both. A and B are both 2-regular graphs with the same number
of nodes, edges, and a single uniform initial colour, and we showed by hand that
refinement reaches a fixed point at round 1 where the colour multiset is identical for
both graphs; this is a structural property (2-regularity plus identical local
neighbourhoods) that no amount of extra layers or hidden units can get around, because
each additional round of message passing is itself an instance of the same
colour-refinement computation. So the bound is architectural, not a training or
capacity limitation.


---
## Task 4. Find a discriminating invariant  &nbsp;&nbsp;

Refinement failed. Something else must succeed, because the two graphs genuinely differ.

Recall the walk-counting theorem: $(A^k)_{ij}$ is the number of walks of length exactly
$k$ from atom $i$ to atom $j$. Therefore $\mathrm{tr}(A^k) = \sum_i (A^k)_{ii}$ counts
**closed** walks of length $k$, that is walks that return to where they started.


In [7]:
# compute the trace of the k-th matrix power for k = 2, 3 and 6.
print(f"{'k':<5}{'tr(A_A^k)':>12}{'tr(A_B^k)':>12}   separates?")
print('-' * 45)
for k in [2, 3, 6]:
    tA = np.trace(np.linalg.matrix_power(A_A, k))
    tB = np.trace(np.linalg.matrix_power(A_B, k))
    print(f'{k:<5}{tA:>12.0f}{tB:>12.0f}   {"YES" if tA != tB else "no"}')


k       tr(A_A^k)   tr(A_B^k)   separates?
---------------------------------------------
2              12          12   no
3               0          12   YES
6             132         132   no


In [8]:
# Optional, not marked: the full adjacency spectra.
# For a conjugated system these are the Huckel orbital energies in units of beta.
print('eigenvalues of A_A:', np.sort(np.linalg.eigvalsh(A_A))[::-1])
print('eigenvalues of A_B:', np.sort(np.linalg.eigvalsh(A_B))[::-1])


eigenvalues of A_A: [ 2.  1.  1. -1. -1. -2.]
eigenvalues of A_B: [ 2.  2. -1. -1. -1. -1.]


### YOUR ANSWER (Task 4)

(a) Which value of $k$ separates A from B?

**k = 3** separates them: tr(A_A^3) = 0 while tr(A_B^3) = 12.

(b) Explain **in terms of walks** why that particular $k$ works. What closed walk exists
in one graph and not the other?

A closed walk of length 3 that revisits distinct edges is exactly a triangle traversed in
one of two directions, so tr(A^3) = 6 × (number of triangles) for a simple undirected
graph (each triangle contributes 3 starting points × 2 directions = 6 closed walks). A is
a 6-cycle and contains **no triangles** at all, so tr(A_A^3) = 0. B is two triangles, so it
contains exactly 2 triangles, giving tr(A_B^3) = 6 × 2 = 12. The closed 3-walk
0 → 1 → 2 → 0 exists in B but has no counterpart in A, since no three atoms in the
6-cycle are mutually bonded.

(c) Explain why the other two values of $k$ fail. Be careful with $k=6$: the result may
not be what you expected, and the explanation is the point of this part.

$k=2$ fails because tr(A^2) = $\sum_i d_i$ = 2·(number of edges) for *any* graph — it only
counts "there and back" walks along single edges, so it depends solely on the edge count,
which is 6 in both A and B by construction; it carries no information about how the edges
are arranged into cycles. $k=6$ is the surprising case: it also **fails** to separate them
(tr(A_A^6) = tr(A_B^6) = 132), even though $k=6$ is large enough in principle to detect
6-cycles. The reason is that closed walks of length 6 get contributions from several
sources at once — walks that go around a hexagon once, walks that go around a triangle
twice, walks that bounce back and forth along single or pairs of edges, etc. — and for
these two particular graphs the totals from all these different walk types happen to sum
to exactly the same number, 132 (which can also be checked from the adjacency spectra:
A's eigenvalues {2,1,1,-1,-1,-2} and B's eigenvalues {2,2,-1,-1,-1,-1} give
$\sum \lambda^6 = 64+1+1+1+1+64 = 132$ for A and $64+64+1+1+1+1=132$ for B). This is a
useful cautionary example: a graph invariant can fail to separate two non-isomorphic
graphs even at a walk length that "should" be informative — cospectral-like coincidences
happen, and you have to check each $k$ rather than assume bigger is always more powerful.

(d) One sentence: what does $\mathrm{tr}(A^2)$ count for *any* graph, and why could it
never have separated these two?

$\mathrm{tr}(A^2)$ counts twice the number of edges (every closed 2-walk is just stepping
out along an edge and immediately back), so it is fixed the moment the edge count is fixed
— and A and B were constructed to have the same number of edges (6), so this invariant was
guaranteed to agree before we even computed it.


---
## Task 5. Propose a fix and defend it  &nbsp;&nbsp;

Computing $\mathrm{tr}(A^k)$ costs $O(n^3)$ and does not transfer cleanly between
molecules of different size. A cheaper repair is to give each atom an extra **input
feature** that already distinguishes the two cases, so that the model separates them at
layer zero without any change to the architecture.

Implement your chosen feature below.


In [9]:
def extra_feature(A):
    """Return a length-n array: one extra scalar feature per atom.

    Feature: ring size, i.e. the number of atoms in the (smallest) ring that
    contains atom i. For a molecule built entirely of disjoint simple rings
    (as here) this is simply the size of the connected component containing
    atom i, found by a linear-time breadth-first search / connected-components
    sweep -- exactly what SSSR ring perception in RDKit or OpenBabel returns.
    """
    n = A.shape[0]
    feature = np.zeros(n)
    visited = np.zeros(n, dtype=bool)
    for start in range(n):
        if visited[start]:
            continue
        # BFS to find the connected component containing `start`
        component = [start]
        visited[start] = True
        frontier = [start]
        while frontier:
            new_frontier = []
            for u in frontier:
                neighbours = np.nonzero(A[u])[0]
                for v in neighbours:
                    if not visited[v]:
                        visited[v] = True
                        component.append(v)
                        new_frontier.append(v)
            frontier = new_frontier
        ring_size = len(component)
        for atom in component:
            feature[atom] = ring_size
    return feature

fA, fB = extra_feature(A_A), extra_feature(A_B)
print('feature on A:', fA)
print('feature on B:', fB)


feature on A: [6. 6. 6. 6. 6. 6.]
feature on B: [3. 3. 3. 3. 3. 3.]


In [10]:
# CHECK: does the feature actually do the job?
assert not np.allclose(np.sort(fA), np.sort(fB)), (
    'Your feature takes the same multiset of values on A and B, '
    'so it cannot separate them. Try again.')
print('The feature separates A from B at the input layer.')

# and does it survive relabelling of the atoms?
perm = np.random.default_rng(0).permutation(6)
A_A_perm = A_A[np.ix_(perm, perm)]
assert np.allclose(np.sort(extra_feature(A_A_perm)), np.sort(fA)), (
    'Your feature changes when the atoms are relabelled. It must be permutation equivariant.')
print('The feature is unchanged by relabelling the atoms, as required.')


The feature separates A from B at the input layer.
The feature is unchanged by relabelling the atoms, as required.


### YOUR ANSWER (Task 5)

(a) Name your feature and state its value on every atom of A and of B.

**Feature: ring size** — the number of atoms in the smallest ring that contains each
atom, computed here (since every atom in both molecules lies in exactly one simple ring)
as the size of the connected component the atom belongs to. On **A** (cyclohexane
skeleton) every atom sits in the single 6-membered ring, so $f_i = 6$ for all
$i = 0,\dots,5$. On **B** (two cyclopropane rings) every atom sits in a 3-membered ring,
so $f_i = 3$ for all atoms. This differs between A and B (all-6 vs all-3), so it separates
them at layer zero, and since it only depends on which connected ring an atom belongs to
and not on how the atoms happen to be numbered, it is unchanged by relabelling
(permutation equivariant).

(b) Justify why it is cheap: what algorithm computes it, and at what cost?

Ring size (more generally, SSSR — the Smallest Set of Smallest Rings) is computed by
standard cheminformatics toolkits, e.g. RDKit's `GetRingInfo()` / OpenBabel's ring
perception, using a breadth-first search or a spanning-tree-plus-fundamental-cycles
algorithm. For molecular graphs, which have bounded valence (degree at most 4 for carbon),
this runs in $O(n)$ time and space, exactly as implemented above with a single BFS sweep
over the atoms — dramatically cheaper than forming $A^k$ and taking an $O(n^3)$ trace.

(c) Name **one further chemical property** that this feature would help a model predict,
and say why.

**Ring strain energy / heat of combustion.** Ring size is the single strongest structural
determinant of angle (Baeyer) strain: 3-membered rings force ~60° C–C–C angles far from
the ideal ~109.5° sp3 angle, while 6-membered rings can adopt an essentially strain-free
chair conformation. A model that can read off ring size directly has immediate access to
the feature that predicts strain energy, reactivity toward ring-opening, and stability —
exactly the property that made A and B so different chemically while being invisible to
plain message passing.

(d) One sentence on the general lesson: when a model provably cannot see something, is
the better response a bigger architecture or a better input feature?

When the limitation is a proven expressivity ceiling (bounded by 1-WL), adding depth or
width is provably useless because every extra layer is still just another round of the
same refinement, so the effective and far cheaper fix is to inject the missing structural
information directly as an input feature (or positional/structural encoding) rather than
to scale up the architecture.


---
## Before you submit

Run the cell below. It confirms only that the notebook executes; it does not mark your
prose answers.


In [11]:
checks = {
    'Task 1: adjacency matrices built': (A_A.sum() == 12 and A_B.sum() == 12
                                          and not np.allclose(A_A, A_B)),
    'Task 2: wl_round implemented':      sorted(wl_round(A_nbutane, [0]*4)) != sorted(wl_round(A_isobutane, [0]*4)),
    'Task 5: extra_feature implemented': not np.allclose(extra_feature(A_A), 0),
}
for k, v in checks.items():
    print(('  OK   ' if v else '  TODO ') + k)
print()
print('Remember: Kernel > Restart & Run All, then save with all output visible.')


  OK   Task 1: adjacency matrices built
  OK   Task 2: wl_round implemented
  OK   Task 5: extra_feature implemented

Remember: Kernel > Restart & Run All, then save with all output visible.
